# AP1: Adjektivcluster

Dieses Notebook leitet die Werte her, die `kapitel7.ipynb` für das Adjektivclustering fest einträgt: Mindesthäufigkeit des Kernvokabulars, Ressource, Repräsentation, Verfahren und Clusterzahl. Gesetzte Werte stehen in der Einstellungszelle. Ergebnis ist die Clusterkarte `ergebnisse_ap1/clusterkarte.csv`, die `kap7_ap2.ipynb` liest.

## Vorbereitung

### Arbeitsverzeichnis

In [2]:
#Projekt-Stammverzeichnis suchen (Ordner mit "Korpus/" oder dem Export)
import os
from pathlib import Path

PROJEKT = Path.cwd()
for _kandidat in [PROJEKT, *PROJEKT.parents]:
    if (_kandidat / "Korpus").is_dir() or (_kandidat / "prototyp_ergebnisse_alle.json").is_file():
        PROJEKT = _kandidat
        break
os.chdir(PROJEKT)
print("Arbeitsverzeichnis:", PROJEKT)

#Eingaben; LIWC-Diktionär und Urteile der Sichtprüfung sind optional
P_EXPORT = Path("prototyp_ergebnisse_alle.json")
P_LIWC = Path("LIWC_German.dic")
P_SICHTPRUEFUNG = Path("kapi6-sichtprüfung.csv")
for pfad in (P_EXPORT, P_LIWC, P_SICHTPRUEFUNG):
    print("vorhanden" if pfad.exists() else "FEHLT    ", "|", pfad)

#Ausgabeordner für Clusterkarte (Eingabe von ap8.ipynb) und Vorlage der Sichtprüfung
AUSGABE = Path("ergebnisse_kapitel7_ap1")
AUSGABE.mkdir(exist_ok=True)

Arbeitsverzeichnis: /home/timc/Workspace/fairytale-personality-profiling/Abgabe
vorhanden | prototyp_ergebnisse_alle.json
vorhanden | LIWC_German.dic
FEHLT     | kapi6-sichtprüfung.csv


### Einstellungen

In [3]:
import json
from collections import Counter, defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import pdist, squareform

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 100)

#Mindesthäufigkeit: gesetzt sind Kandidaten und Kriterium, der Wert folgt daraus
BELEGSTUFEN = [1, 2, 3, 5, 10]       #Mindestzahl der Belege je Normform
MAERCHENSTUFEN = [1, 2, 3]           #Mindestzahl verschiedener Märchen
SAMMLUNGSSTUFEN = [1, 2]             #Mindestzahl verschiedener Sammlungen

#Ressourcen
LIWC_MIN_STAMMLAENGE = 3             #Mindestbuchstaben; kürzere LIWC-Stämme sind sonst zu breit (zB "gu*" trifft zu viel)

#Repräsentationsvergleich: gesetzt sind Evaluationsmengen und Entscheidungsregel
MAX_SYNSETGROESSE = 8                #Maximal Synsets bis acht Wörterr
MIN_FIGUREN_JE_FORM = 1              #Später für Repräsentation A: jede Form mit mindestens einer Figur wird gezählt
ZUFALL_STARTWERT = 42                #Seed für: Startwert, Ziehungen, Dimensionen
ANKER_ZIEHUNGEN = 25
ANKER_DIMENSIONEN = 100

#Clusterbildung: gesetzt sind Kandidaten und Kriterium, Verfahren und Clusterzahl folgen daraus
LINKAGE_KANDIDATEN = ["average", "complete", "ward"]
SCHNITTPUNKTE = 120                  #geprüfte Schnitthöhen je Verfahren
MAX_SINGLETON_ANTEIL = 0.25          #höchstens ein Viertel der Formen steht allein
MAX_GROESSTES_CLUSTER = 0.25         #höchstens ein Viertel der Formen im größten Cluster
MIN_CLUSTER, MAX_CLUSTER = 10, 60    #zulässige Clusterzahl

#Benennung (gesetzt)
MIN_MITGLIEDER_FUER_RESSOURCE = 2    #ein Mitglied begründet keine Kategorie
MIN_DECKUNG_RESSOURCE = 0.34         #die Grundlage muss ein Drittel des Clusters tragen
MEHRHEIT_N = 3                       #Mehrheitsregel: die drei häufigsten Normformen


def prozent(teil, ganzes):
    #Anteil in Prozent mit einer Nachkommastelle
    return round(100 * teil / ganzes, 1)

### Sprachmodell und OdeNet

In [4]:
import spacy
import wn

#Statische Wortvektoren aus de_core_news_lg
nlp = spacy.load("de_core_news_lg", disable=["parser", "ner", "tagger", "attribute_ruler", "lemmatizer"])
print("spaCy-Modell:", nlp.meta["lang"] + "_" + nlp.meta["name"], nlp.meta["version"])

#OdeNet 1.4 über die Bibliothek wn; beim ersten Lauf wird es heruntergeladen
try:
    de = wn.Wordnet("odenet:1.4")
except wn.Error:
    wn.download("odenet:1.4")
    de = wn.Wordnet("odenet:1.4")
print("OdeNet:", de.lexicons()[0].version)


def hole_vektor(wort):
    """Statischer Vektor einer Form; probiert Original- und Kleinschreibung."""
    for kandidat in (wort, wort.lower(), wort.capitalize()):
        if nlp.vocab.has_vector(kandidat):
            return nlp.vocab.get_vector(kandidat)
    return None


def adjektiv_synsets(lemma):
    """Synsets eines Lemmas, beschränkt auf adjektivische Wortarten."""
    return [ss for ss in de.synsets(lemma) if ss.pos in ("a", "s")]

spaCy-Modell: de_core_news_lg 3.8.0
OdeNet: 1.4


### Export laden

In [5]:
with open(P_EXPORT, encoding="utf-8") as f:
    daten = json.load(f)

zeilen_alle = daten["adjektiv_zuordnungen_alle"]
norm_karte = daten["adjektiv_norm_karte"]
korpus_meta = daten["korpus"]

#Grundgesamtheit: zugeordnete Belege mit Normform
zugeordnet = [z for z in zeilen_alle if z.get("status") == "zugeordnet" and z.get("lemma_norm")]

print("Pipeline-Version:", daten.get("meta", {}).get("notebook_version"))
print("Adjektivbelege:", len(zeilen_alle), "| zugeordnet:", len(zugeordnet))

Pipeline-Version: V1.0
Adjektivbelege: 11022 | zugeordnet: 4254


## 1 Mindesthäufigkeit

Normformen mit Belegzahl und Streuung über Märchen und Sammlungen.

In [6]:
#Normformen mit Belegzahl, Zahl der Märchen (Normformkarte) und Zahl der Sammlungen (Zuordnungszeilen)
sammlungen_je_norm = defaultdict(set)
for z in zugeordnet:
    sammlungen_je_norm[z["lemma_norm"]].add(z.get("sammlung") or "(Wurzel)")
df_norm = pd.DataFrame([{"normform": n, "belege": e["belege"], "anzahl_maerchen": e["anzahl_maerchen"],
                         "anzahl_sammlungen": len(sammlungen_je_norm.get(n, ()))}
                        for n, e in norm_karte.items()])
df_norm = df_norm.sort_values(["belege", "normform"], ascending=[False, True]).reset_index(drop=True)
n_belege = int(df_norm["belege"].sum())
einmal = int((df_norm["belege"] == 1).sum())
print("Normformen:", len(df_norm), "| Belege:", n_belege)
print("Normformen mit nur einem Beleg:", einmal, "|", prozent(einmal, len(df_norm)), "% des Vokabulars |",
      prozent(einmal, n_belege), "% der Belege")

Normformen: 901 | Belege: 4254
Normformen mit nur einem Beleg: 508 | 56.4 % des Vokabulars | 11.9 % der Belege


### EXKURS - Schwellen für Belegverlust und Formbegrenzung ausrechnen

In [7]:
MAX_BELEGVERLUST = 0.167     #Gewählt wurde höchstens 1/6 der Belegmasse
MIN_DIMENSIONSABBAU = 0.50   #Gewählt wurde mindestens die Hälfte der Formen

#Hauptexport im aktuellen Ordner oder darüber suchen
pfad = Path("prototyp_ergebnisse_alle.json")

daten = json.loads(pfad.read_text(encoding="utf-8"))
belege = [z for z in daten["adjektiv_zuordnungen_alle"] if z.get("status") == "zugeordnet"]


def prozent_de(anteil):
    """Formatiert einen Anteil deutsch, z. B. 0.575 -> '57,5 %'."""
    return format(anteil * 100, ".1f").replace(".", ",") + " %"


for feld, stand in [("lemma", "vor Normalisierung"), ("lemma_norm", "nach Normalisierung")]:
    haeufigkeit = Counter(z[feld] for z in belege if z.get(feld))
    formen = len(haeufigkeit)
    belege_gesamt = sum(haeufigkeit.values())
    einmalformen = sum(1 for n in haeufigkeit.values() if n == 1)

    #Jede Einmalform trägt genau einen Beleg, deshalb selber Zähler
    formanteil = einmalformen / formen
    beleganteil = einmalformen / belege_gesamt

    print(stand, "(" + feld + "):", einmalformen, "von", formen, "Formen einmal belegt,", belege_gesamt, "Belege")
    print("  Formanteil ", prozent_de(formanteil), " >= Haelfte ", prozent_de(MIN_DIMENSIONSABBAU) + ":", formanteil >= MIN_DIMENSIONSABBAU)
    print("  Beleganteil", prozent_de(beleganteil), " <= Sechstel", prozent_de(MAX_BELEGVERLUST) + ":", beleganteil <= MAX_BELEGVERLUST)

vor Normalisierung (lemma): 573 von 978 Formen einmal belegt, 4254 Belege
  Formanteil  58,6 %  >= Haelfte  50,0 %: True
  Beleganteil 13,5 %  <= Sechstel 16,7 %: True
nach Normalisierung (lemma_norm): 508 von 901 Formen einmal belegt, 4254 Belege
  Formanteil  56,4 %  >= Haelfte  50,0 %: True
  Beleganteil 11,9 %  <= Sechstel 16,7 %: True


### EXKURS - Ende

Raster der Kandidatenschwellen. Zulässig ist höchstens ein Sechstel Belegverlust bei mindestens der Hälfte Dimensionsabbau; gewählt => Kombination mit der meisten Belegmasse

In [8]:
#Raster der Kandidatenschwellen: Dimensionsabbau und Belegverlust je Kombination
zeilen = []
for b in BELEGSTUFEN:
    for m in MAERCHENSTUFEN:
        for s in SAMMLUNGSSTUFEN:
            rest = df_norm[(df_norm["belege"] >= b) & (df_norm["anzahl_maerchen"] >= m)
                           & (df_norm["anzahl_sammlungen"] >= s)]
            belege = int(rest["belege"].sum())
            zeilen.append({"min_belege": b, "min_maerchen": m, "min_sammlungen": s,
                           "normformen": len(rest), "dimensionsabbau_%": prozent(len(df_norm) - len(rest), len(df_norm)),
                           "belege": belege, "belegverlust_%": prozent(n_belege - belege, n_belege),
                           "zulaessig": (1 - belege / n_belege <= MAX_BELEGVERLUST
                                         and 1 - len(rest) / len(df_norm) >= MIN_DIMENSIONSABBAU)})
raster = pd.DataFrame(zeilen)

#Gewählt wird die zulässige Kombination mit der meisten Belegmasse
wahl = raster[raster["zulaessig"]].sort_values(["belege", "dimensionsabbau_%"], ascending=[False, False]).iloc[0]
MIN_BELEGE = int(wahl["min_belege"])
maske = ((df_norm["belege"] >= MIN_BELEGE) & (df_norm["anzahl_maerchen"] >= int(wahl["min_maerchen"]))
         & (df_norm["anzahl_sammlungen"] >= int(wahl["min_sammlungen"])))
df_kern = df_norm[maske].reset_index(drop=True)
KERN = list(df_kern["normform"])
kern_menge = set(KERN)
HAEUFIGKEIT = dict(zip(df_kern["normform"], df_kern["belege"]))
n_kern_belege = int(df_kern["belege"].sum())
print("Gewählt: mindestens", MIN_BELEGE, "Belege,", int(wahl["min_maerchen"]), "Märchen,",
      int(wahl["min_sammlungen"]), "Sammlung")
print("Kernvokabular:", len(KERN), "Normformen mit", n_kern_belege, "Belegen")
raster

Gewählt: mindestens 2 Belege, 1 Märchen, 1 Sammlung
Kernvokabular: 393 Normformen mit 3746 Belegen


,min_belege,min_maerchen,min_sammlungen,normformen,dimensionsabbau_%,belege,belegverlust_%,zulaessig
0,1,1,1,901,0.0,4254,0.0,False
1,1,1,2,286,68.3,3359,21.0,False
2,1,2,1,362,59.8,3669,13.8,True
3,1,2,2,286,68.3,3359,21.0,False
4,1,3,1,215,76.1,3277,23.0,False
5,1,3,2,195,78.4,3152,25.9,False
6,2,1,1,393,56.4,3746,11.9,True
7,2,1,2,286,68.3,3359,21.0,False
8,2,2,1,362,59.8,3669,13.8,True
9,2,2,2,286,68.3,3359,21.0,False


## 2 Ressource

Abdeckung des Kernvokabulars durch LIWC2001 und OdeNet.

In [9]:
def lies_liwc_dic(pfad):
    """Liest ein LIWC-Diktionär im .dic-Format (Legende zwischen zwei '%'-Zeilen)."""
    zeilen = Path(pfad).read_text(encoding="utf-8", errors="replace").splitlines()
    marker = [i for i, z in enumerate(zeilen) if z.strip() == "%"]
    kategorien = {}
    for z in zeilen[marker[0] + 1:marker[1]]:
        teile = z.split()
        if len(teile) >= 2:
            kategorien[teile[0]] = " ".join(teile[1:])
    exakt, staemme = {}, {}
    for z in zeilen[marker[1] + 1:]:
        teile = z.split()
        if not teile:
            continue
        wort = teile[0].lower()
        nummern = [t for t in teile[1:] if t.isdigit()]
        if not nummern:
            continue
        namen = tuple(kategorien.get(n, "?" + n) for n in nummern)
        if wort.endswith("*"):
            staemme[wort[:-1]] = namen
        else:
            exakt[wort] = namen
    return {"kategorien": kategorien, "exakt": exakt, "staemme": staemme}


def liwc_treffer(wort, diktionaer, min_stammlaenge=3):
    """Kategorien eines Wortes; der längste passende Stamm gewinnt, leer heißt nicht abgedeckt."""
    w = (wort or "").lower()
    if w in diktionaer["exakt"]:
        return diktionaer["exakt"][w]
    for laenge in range(len(w), min_stammlaenge - 1, -1):
        if w[:laenge] in diktionaer["staemme"]:
            return diktionaer["staemme"][w[:laenge]]
    return ()


vergleich = []
#LIWC2001 ist lizenzpflichtig; fehlt das Diktionär, entfällt dieser Teil
if P_LIWC.exists():
    liwc = lies_liwc_dic(P_LIWC)
    kategorien_je_form = {f: liwc_treffer(f, liwc, LIWC_MIN_STAMMLAENGE) for f in KERN}
    kategorien_je_form = {f: k for f, k in kategorien_je_form.items() if k}
    belege_liwc = sum(HAEUFIGKEIT[f] for f in kategorien_je_form)
    vergleich.append({"Ressource": "LIWC2001", "Normformen": len(kategorien_je_form),
                      "Normformen %": prozent(len(kategorien_je_form), len(KERN)),
                      "Belege": belege_liwc, "Belege %": prozent(belege_liwc, n_kern_belege),
                      "Synsets mit Oberbegriff %": None})
    zaehler = Counter(k for ks in kategorien_je_form.values() for k in set(ks))
    print("Häufigste LIWC-Kategorien (Anteil an den abgedeckten Normformen):")
    for kategorie, anzahl in zaehler.most_common(6):
        print("  ", kategorie, anzahl, "|", prozent(anzahl, len(kategorien_je_form)), "%")
else:
    print("LIWC-Diktionär nicht gefunden:", P_LIWC)

#OdeNet: Abdeckung und Anteil der Adjektivsynsets mit Oberbegriff (Weg der Benennung)
odenet_formen, synsets_gesamt, mit_oberbegriff = [], 0, 0
for form in KERN:
    sss = adjektiv_synsets(form)
    if sss:
        odenet_formen.append(form)
    synsets_gesamt += len(sss)
    mit_oberbegriff += sum(1 for ss in sss if ss.hypernyms())
belege_odenet = sum(HAEUFIGKEIT[f] for f in odenet_formen)
vergleich.append({"Ressource": "OdeNet " + de.lexicons()[0].version, "Normformen": len(odenet_formen),
                  "Normformen %": prozent(len(odenet_formen), len(KERN)),
                  "Belege": belege_odenet, "Belege %": prozent(belege_odenet, n_kern_belege),
                  "Synsets mit Oberbegriff %": prozent(mit_oberbegriff, synsets_gesamt)})
print("Adjektivsynsets mit Oberbegriff:", mit_oberbegriff, "von", synsets_gesamt)

#Gewählt wird die Ressource mit der höheren Abdeckung nach Normformen und Belegmasse
df_ressourcen = pd.DataFrame(vergleich)
RESSOURCE = df_ressourcen.sort_values(["Normformen %", "Belege %"], ascending=False).iloc[0]["Ressource"]
print("Gewählte Ressource:", RESSOURCE)
df_ressourcen

Häufigste LIWC-Kategorien (Anteil an den abgedeckten Normformen):
   Affect 134 | 57.8 %
   Positiveemotion 70 | 30.2 %
   Negativeemotion 64 | 27.6 %
   Cognitivemechanism 26 | 11.2 %
   Physical 23 | 9.9 %
   Body 18 | 7.8 %
Adjektivsynsets mit Oberbegriff: 47 von 1112
Gewählte Ressource: OdeNet 1.4


,Ressource,Normformen,Normformen %,Belege,Belege %,Synsets mit Oberbegriff %
0,LIWC2001,232,59.0,2716,72.5,NaN
1,OdeNet 1.4,333,84.7,3282,87.6,4.2


## 3 Repräsentation

Figurenverteilung (A) gegen Wortvektoren (B): AUC der strengen Synonympaare gegen beliebige Paare mit Intervall nach Hanley und McNeil. Der Zufallsanker muss 0,5 umschließen.

In [10]:
#Evaluationsmengen aus OdeNet: Normformen im selben Synset (Synonyme) und Antonympaare
mitglieder_synset = defaultdict(set)
for form in KERN:
    for ss in adjektiv_synsets(form):
        mitglieder_synset[ss.id].add(form)


def paare_aus_synsets(max_groesse=None):
    """Alle Paare, die mindestens ein Synset teilen."""
    paare = set()
    for menge in mitglieder_synset.values():
        if max_groesse is None or len(menge) <= max_groesse:
            paare.update(combinations(sorted(menge), 2))
    return paare


SYNONYM_ALLE = paare_aus_synsets()
SYNONYM_STRENG = paare_aus_synsets(MAX_SYNSETGROESSE)
ANTONYM = set()
for form in KERN:
    for ss in de.synsets(form):
        for gegen in ss.get_related("antonym"):
            for g in gegen.lemmas():
                g = g.lower()
                if g in kern_menge and g != form:
                    ANTONYM.add(tuple(sorted((form, g))))
#Ein Paar, das zugleich synonym und antonym ist, taugt für keine der Mengen
strittig = SYNONYM_ALLE & ANTONYM
SYNONYM_ALLE -= strittig
SYNONYM_STRENG -= strittig
ANTONYM -= strittig
print("Synonympaare:", len(SYNONYM_ALLE), "| streng (Synsets bis", MAX_SYNSETGROESSE, "Mitglieder):",
      len(SYNONYM_STRENG), "| Antonympaare:", len(ANTONYM))

#Repräsentation A: Verteilung jeder Normform über die Figuren (TF-IDF, längennormiert)
form_figur = Counter((z["lemma_norm"], z["figur"]) for z in zugeordnet
                     if z["lemma_norm"] in kern_menge and z.get("figur"))
figuren_a = sorted({f for _, f in form_figur})
figuren_je_form = Counter(form for form, _ in form_figur)
FORMEN_A = [f for f in KERN if figuren_je_form[f] >= MIN_FIGUREN_JE_FORM]
INDEX_A = {f: i for i, f in enumerate(FORMEN_A)}
spalte_von = {f: j for j, f in enumerate(figuren_a)}
M_A = np.zeros((len(FORMEN_A), len(figuren_a)))
for (form, figur), n in form_figur.items():
    M_A[INDEX_A[form], spalte_von[figur]] = n
M_A = M_A * (np.log((len(FORMEN_A) + 1) / ((M_A > 0).sum(axis=0) + 1)) + 1.0)
M_A = M_A / np.linalg.norm(M_A, axis=1, keepdims=True)

#Repräsentation B: statische Wortvektoren
FORMEN_B, vektoren_b, OHNE_VEKTOR = [], [], []
for form in KERN:
    v = hole_vektor(form)
    if v is not None and np.linalg.norm(v) > 0:
        FORMEN_B.append(form)
        vektoren_b.append(v)
    else:
        OHNE_VEKTOR.append(form)
M_B = np.vstack(vektoren_b)
M_B = M_B / np.linalg.norm(M_B, axis=1, keepdims=True)
INDEX_B = {f: i for i, f in enumerate(FORMEN_B)}
print("A:", len(FORMEN_A), "Normformen x", len(figuren_a), "Figuren | B:", len(FORMEN_B), "Normformen mit Wortvektor")


def auc_rangbasiert(positive_werte, negative_werte):
    """Fläche unter der ROC-Kurve über Ränge (Mann-Whitney-U), Bindungen gemittelt."""
    p = np.asarray(positive_werte, dtype=float)
    n = np.asarray(negative_werte, dtype=float)
    alle = np.concatenate([p, n])
    ordnung = alle.argsort(kind="mergesort")
    raenge = np.empty(len(alle))
    raenge[ordnung] = np.arange(1, len(alle) + 1, dtype=float)
    sortiert = alle[ordnung]
    i = 0
    while i < len(sortiert):
        j = i
        while j + 1 < len(sortiert) and sortiert[j + 1] == sortiert[i]:
            j += 1
        if j > i:
            raenge[ordnung[i:j + 1]] = (i + j + 2) / 2.0
        i = j + 1
    return float((raenge[:len(p)].sum() - len(p) * (len(p) + 1) / 2.0) / (len(p) * len(n)))


def auc_intervall(auc, n_positiv, n_negativ, z=1.96):
    """95-Prozent-Intervall der ROC-Fläche nach Hanley und McNeil (1982)."""
    q1 = auc / (2.0 - auc)
    q2 = 2.0 * auc ** 2 / (1.0 + auc)
    varianz = ((auc * (1 - auc) + (n_positiv - 1) * (q1 - auc ** 2)
                + (n_negativ - 1) * (q2 - auc ** 2)) / (n_positiv * n_negativ))
    se = float(np.sqrt(max(varianz, 0.0)))
    return max(0.0, auc - z * se), min(1.0, auc + z * se)


#Vergleich auf denselben Normformen: strenge Synonympaare gegen beliebige Paare
VERGLEICH = sorted(set(FORMEN_A) & set(FORMEN_B))
alle_paare = list(combinations(VERGLEICH, 2))
synonym_streng = SYNONYM_STRENG & set(alle_paare)
antonym = ANTONYM & set(alle_paare)
nicht_beliebig = (SYNONYM_ALLE | ANTONYM) & set(alle_paare)
beliebig = [p for p in alle_paare if p not in nicht_beliebig]
print("Vergleich auf", len(VERGLEICH), "Normformen:", len(synonym_streng), "Synonympaare,", len(antonym),
      "Antonympaare,", len(beliebig), "beliebige Paare")


def messe(M_, index):
    """AUC der strengen Synonympaare und der Antonympaare gegen die beliebigen Paare."""
    ia = np.array([index[a] for a, _ in alle_paare])
    ib = np.array([index[b] for _, b in alle_paare])
    kosinus = dict(zip(alle_paare, (M_[ia] * M_[ib]).sum(axis=1)))
    rest = [kosinus[p] for p in beliebig]
    return (auc_rangbasiert([kosinus[p] for p in synonym_streng], rest),
            auc_rangbasiert([kosinus[p] for p in antonym], rest))


zeilen = []
for name, M_, index in (("A Verteilung über die Figuren", M_A, INDEX_A), ("B Wortvektoren", M_B, INDEX_B)):
    auc, auc_antonym = messe(M_, index)
    unten, oben = auc_intervall(auc, len(synonym_streng), len(beliebig))
    zeilen.append({"Repräsentation": name, "AUC": auc, "unten": unten, "oben": oben, "AUC Antonyme": auc_antonym})

#Zufallsanker: normalverteilte Vektoren, gemittelt über 25 Ziehungen; unten und oben bezeichnen die Spannweite
rng = np.random.default_rng(ZUFALL_STARTWERT)
INDEX_Z = {f: i for i, f in enumerate(VERGLEICH)}
anker_auc = []
for _ in range(ANKER_ZIEHUNGEN):
    M_Z = rng.normal(size=(len(VERGLEICH), ANKER_DIMENSIONEN))
    M_Z = M_Z / np.linalg.norm(M_Z, axis=1, keepdims=True)
    anker_auc.append(messe(M_Z, INDEX_Z)[0])
zeilen.append({"Repräsentation": "Zufallsanker (" + str(ANKER_ZIEHUNGEN) + " Ziehungen)",
               "AUC": float(np.mean(anker_auc)), "unten": float(np.min(anker_auc)), "oben": float(np.max(anker_auc)),
               "AUC Antonyme": None})
df_repraesentation = pd.DataFrame(zeilen)

#Entscheidungsregel: höhere AUC auf den strengen Synonympaaren, sofern der Anker 0,5 umschließt
anker_ok = min(anker_auc) <= 0.5 <= max(anker_auc)
kandidaten = df_repraesentation.iloc[:2].sort_values("AUC", ascending=False)
GEWAEHLT = kandidaten.iloc[0]["Repräsentation"]
print("Zufallsanker umschließt 0,5:", "ja" if anker_ok else "nein, die Messung ist zu prüfen")
print("Gewählt:", GEWAEHLT, "| Intervalle überlappen:",
      "ja" if kandidaten.iloc[0]["unten"] <= kandidaten.iloc[1]["oben"] else "nein")
df_repraesentation.round(4)

Synonympaare: 535 | streng (Synsets bis 8 Mitglieder): 456 | Antonympaare: 163
A: 393 Normformen x 501 Figuren | B: 380 Normformen mit Wortvektor
Vergleich auf 380 Normformen: 453 Synonympaare, 163 Antonympaare, 71315 beliebige Paare
Zufallsanker umschließt 0,5: ja
Gewählt: B Wortvektoren | Intervalle überlappen: nein


,Repräsentation,AUC,unten,oben,AUC Antonyme
0,A Verteilung über die Figuren,0.5842,0.5565,0.6118,0.5816
1,B Wortvektoren,0.7991,0.7745,0.8238,0.7043
2,Zufallsanker (25 Ziehungen),0.4963,0.4717,0.5167,NaN


## 4 Verfahren und Clusterzahl

Average, complete und Ward unter demselben Kriterium: höchstens ein Viertel allein stehender Formen, höchstens ein Viertel im größten Cluster, 10 bis 60 Cluster. Gewählt wird die höchste Silhouette unter den zulässigen Zerlegungen.

In [11]:
#Geclustert wird auf der gewählten Repräsentation (längennormierte Vektoren)
FORMEN, M = (FORMEN_B, M_B) if GEWAEHLT == "B Wortvektoren" else (FORMEN_A, M_A)
D = squareform(np.clip(pdist(M, metric="cosine"), 0.0, 2.0))


def silhouette(D_, zuordnung):
    """Mittlerer Silhouettenwert auf der Kosinusdistanz; Formen in Einzelclustern zählen null."""
    zuordnung = np.asarray(zuordnung)
    gruppen = {c: np.where(zuordnung == c)[0] for c in set(zuordnung.tolist())}
    if len(gruppen) < 2:
        return 0.0
    werte = []
    for i, c in enumerate(zuordnung):
        eigene = gruppen[c][gruppen[c] != i]
        if len(eigene) == 0:
            werte.append(0.0)
            continue
        a = D_[i, eigene].mean()
        b = min(D_[i, idx].mean() for fremd, idx in gruppen.items() if fremd != c)
        werte.append((b - a) / max(a, b) if max(a, b) > 0 else 0.0)
    return float(np.mean(werte))


def verkettung(M_, methode):
    """Ward rechnet euklidisch (bei Länge eins monoton zur Kosinusdistanz), die übrigen auf der Kosinusdistanz."""
    if methode == "ward":
        return linkage(pdist(M_, metric="euclidean"), method="ward")
    return linkage(np.clip(pdist(M_, metric="cosine"), 0.0, 2.0), method=methode)


#Je Verfahren 120 Schnitthöhen; zulässig nach dem gesetzten Kriterium
kurven, verkettungen = {}, {}
for methode in LINKAGE_KANDIDATEN:
    Z_ = verkettung(M, methode)
    hoehe = float(Z_[:, 2].max())
    zeilen = []
    for schwelle in np.linspace(0.02 * hoehe, 0.999 * hoehe, SCHNITTPUNKTE):
        zu = fcluster(Z_, t=float(schwelle), criterion="distance")
        groessen = Counter(zu.tolist())
        zeilen.append({"schwelle": float(schwelle), "cluster": len(groessen),
                       "singleton_anteil": sum(1 for n in groessen.values() if n == 1) / len(zu),
                       "groesstes_cluster": max(groessen.values()) / len(zu),
                       "silhouette": silhouette(D, zu)})
    kurve = pd.DataFrame(zeilen)
    kurve["zulaessig"] = ((kurve["singleton_anteil"] <= MAX_SINGLETON_ANTEIL)
                          & (kurve["groesstes_cluster"] <= MAX_GROESSTES_CLUSTER)
                          & kurve["cluster"].between(MIN_CLUSTER, MAX_CLUSTER))
    kurven[methode], verkettungen[methode] = kurve, Z_

#Je Verfahren die zulässige Zerlegung mit der höchsten Silhouette; gewählt wird das beste Verfahren
beste = {m: k[k["zulaessig"]].loc[k[k["zulaessig"]]["silhouette"].idxmax()]
         for m, k in kurven.items() if k["zulaessig"].any()}
LINKAGE = max(beste, key=lambda m: beste[m]["silhouette"])
wahl = beste[LINKAGE]
SCHWELLE = float(wahl["schwelle"])
K = int(wahl["cluster"])
Z = verkettungen[LINKAGE]
frei = kurven[LINKAGE].loc[kurven[LINKAGE]["silhouette"].idxmax()]
print("Gewählt:", LINKAGE, "|", K, "Cluster | Silhouette", format(wahl["silhouette"], ".4f"),
      "| Schwelle", format(SCHWELLE, ".4f"), "| allein stehend", prozent(wahl["singleton_anteil"], 1),
      "% | größtes Cluster", prozent(wahl["groesstes_cluster"], 1), "%")
print("Ohne die Regeln:", int(frei["cluster"]), "Cluster | Silhouette", format(frei["silhouette"], ".4f"))

pd.DataFrame([{"Verfahren": m, "zulässige Schnitte": int(k["zulaessig"].sum()),
               "Cluster": int(beste[m]["cluster"]) if m in beste else None,
               "Silhouette": round(float(beste[m]["silhouette"]), 4) if m in beste else None,
               "Schwelle": round(float(beste[m]["schwelle"]), 4) if m in beste else None}
              for m, k in kurven.items()])

Gewählt: ward | 59 Cluster | Silhouette 0.1000 | Schwelle 1.4434 | allein stehend 0.0 % | größtes Cluster 4.5 %
Ohne die Regeln: 150 Cluster | Silhouette 0.1502


,Verfahren,zulässige Schnitte,Cluster,Silhouette,Schwelle
0,average,0,NaN,NaN,NaN
1,complete,27,57.0,0.0938,0.7599
2,ward,27,59.0,0.1000,1.4434


## 5 Partition und Benennung

Benennung in drei Stufen: Oberbegriff, gemeinsames Synset, Mehrheitsregel. Die Clusterkarte geht an `kap7_ap2.ipynb`.

In [12]:
#Partition: Mitglieder je Cluster, nach Belegzahl geordnet
zuordnung = fcluster(Z, t=SCHWELLE, criterion="distance")
CLUSTER_JE_FORM = {f: int(c) for f, c in zip(FORMEN, zuordnung)}
MITGLIEDER = defaultdict(list)
for form, c in CLUSTER_JE_FORM.items():
    MITGLIEDER[c].append(form)
MITGLIEDER = {c: sorted(m, key=lambda f: (-HAEUFIGKEIT.get(f, 0), f)) for c, m in MITGLIEDER.items()}

#Adjektivische Oberbegriffe und Synsets je Normform aus OdeNet
OBERBEGRIFFE, SYNSET_MITGLIEDER = {}, {}
for form in FORMEN:
    oben = set()
    for ss in adjektiv_synsets(form):
        SYNSET_MITGLIEDER.setdefault(ss.id, set()).add(form)
        for hyper in ss.hypernyms():
            if hyper.pos not in ("a", "s"):
                continue
            for lemma in hyper.lemmas():
                if " " not in lemma:
                    oben.add(lemma.lower())
    if oben:
        OBERBEGRIFFE[form] = oben


def traegt(n, gesamt):
    """Trägt eine Grundlage aus OdeNet den Namen des ganzen Clusters?"""
    return n >= MIN_MITGLIEDER_FUER_RESSOURCE and n / gesamt >= MIN_DECKUNG_RESSOURCE


def waehle_namen(mitglieder):
    """Dreistufige Kaskade: Oberbegriff, gemeinsames Synset, Mehrheitsregel."""
    menge = set(mitglieder)
    gesamt = len(menge)
    #1. Oberbegriff, den genügend Mitglieder teilen
    zaehler = Counter(ob for form in menge for ob in OBERBEGRIFFE.get(form, ()))
    geteilt = [(ob, n) for ob, n in zaehler.items() if traegt(n, gesamt)]
    if geteilt:
        ob, n = max(geteilt, key=lambda p: (p[1], -len(p[0])))
        return ob.upper(), "oberbegriff"
    #2. Synset mit den meisten Mitgliedern; Etikett ist dessen häufigste Normform im Cluster
    beste_id, beste_zahl = None, 0
    for sid, formen in SYNSET_MITGLIEDER.items():
        gemeinsam = len(formen & menge)
        if gemeinsam > beste_zahl:
            beste_id, beste_zahl = sid, gemeinsam
    if traegt(beste_zahl, gesamt):
        im_synset = sorted(SYNSET_MITGLIEDER[beste_id] & menge, key=lambda f: (-HAEUFIGKEIT.get(f, 0), f))
        return im_synset[0].upper(), "synset"
    #3. Mehrheitsregel: die drei häufigsten Normformen
    fuehrend = sorted(menge, key=lambda f: (-HAEUFIGKEIT.get(f, 0), f))[:MEHRHEIT_N]
    return "/".join(fuehrend).upper(), "mehrheitsregel"


#Namen vergeben, größte Cluster zuerst; Namensgleichheit wird durchnummeriert
zeilen, vergeben = [], Counter()
for c in sorted(MITGLIEDER, key=lambda x: -len(MITGLIEDER[x])):
    name, stufe = waehle_namen(MITGLIEDER[c])
    vergeben[name] += 1
    if vergeben[name] > 1:
        name = name + "-" + str(vergeben[name])
    zeilen.append({"cluster": c, "name": name, "stufe": stufe, "formen": len(MITGLIEDER[c]),
                   "belege": sum(HAEUFIGKEIT.get(f, 0) for f in MITGLIEDER[c]),
                   "mitglieder": ", ".join(MITGLIEDER[c])})
df_cluster = pd.DataFrame(zeilen)
NAME_JE_CLUSTER = dict(zip(df_cluster["cluster"], df_cluster["name"]))
stufen = df_cluster["stufe"].value_counts()
print("Benennung über Oberbegriff:", int(stufen.get("oberbegriff", 0)), "| Synset:", int(stufen.get("synset", 0)),
      "| Mehrheitsregel:", int(stufen.get("mehrheitsregel", 0)))

#Clusterkarte für ap8.ipynb: eine Zeile je Normform mit Clusternummer und Clustername
P_CLUSTERKARTE = AUSGABE / "clusterkarte.csv"
pd.DataFrame([{"normform": f, "cluster": c, "name": NAME_JE_CLUSTER[c]} for f, c in CLUSTER_JE_FORM.items()]
             ).to_csv(P_CLUSTERKARTE, index=False, encoding="utf-8")
print("Geschrieben:", P_CLUSTERKARTE, "|", len(CLUSTER_JE_FORM), "Normformen in", len(MITGLIEDER), "Clustern")
df_cluster.sort_values("belege", ascending=False).reset_index(drop=True)

Benennung über Oberbegriff: 2 | Synset: 14 | Mehrheitsregel: 43
Geschrieben: ergebnisse_kapitel7_ap1/clusterkarte.csv | 380 Normformen in 59 Clustern


,cluster,name,stufe,formen,belege,mitglieder
0,28,KLEIN,synset,3,351,"klein, groß, winzig"
1,41,ALTE/JUNG/BETAGT,mehrheitsregel,5,331,"alte, jung, betagt, rüstig, schwarzhaarig"
2,9,GUT/GENAU/FALSCH,mehrheitsregel,10,180,"gut, genau, falsch, voll, vorsichtig, sicher, klar, richtig, vernünftig, ordentlich"
3,46,ARM/BÖS/DUMM,mehrheitsregel,10,176,"arm, bös, dumm, elend, wehe, arg, faul, lahm, leid, übel"
4,20,SCHÖN/HÜBSCH/FEIN,mehrheitsregel,6,171,"schön, hübsch, fein, süß, edel, glänzend"
5,30,LIEB/GLÜCKLICH/EINSAM,mehrheitsregel,6,165,"lieb, glücklich, einsam, geliebt, unglücklich, wohlhabend"
6,53,LUSTIG/TRAURIG/GARSTIG,mehrheitsregel,13,122,"lustig, traurig, garstig, boshaft, grausam, bitterböse, frech, hartherzig, rachsüchtig, schlau, verflucht, verrucht, verrückt"
7,29,FREUNDLICH/HERRLICH/LIEBLICH,mehrheitsregel,17,107,"freundlich, herrlich, lieblich, fröhlich, niedlich, vergnügen, wunderbar, reizend, wunderlich, himmlisch, köstlich, liebenswürdig, zärtlich, allerliebst, entzücken, verzaubert, wunderschön"
8,2,ANDERER/EIGEN/SORGFÄLTIG,mehrheitsregel,12,95,"anderer, eigen, sorgfältig, gerecht, verständig, inständig, sämtlich, unmittelbar, allgemein, aufrecht, beständig, vertraulich"
9,32,HEILIG/GOTTLOS/FROMM,mehrheitsregel,10,91,"heilig, gottlos, fromm, gnädig, gehorsam, demütig, gottesfürchtig, gütig, heil, tugendhaft"


Vorlage der Sichtprüfung und Abgleich mit den vorhandenen Urteilen.

In [13]:
#Vorlage der Sichtprüfung (Vollerhebung): Mitglieder alphabetisch, Urteilsspalten leer, Namen getrennt
vorlage = pd.DataFrame([{"lfd_nr": c, "cluster_id": c, "mitglieder": ", ".join(sorted(MITGLIEDER[c])),
                         "nicht_zugehoerig": "", "frage1": "", "frage2": "", "notiz": "", "beurteiler": "", "datum": ""}
                        for c in sorted(MITGLIEDER)])
vorlage.to_csv(AUSGABE / "sichtpruefung_vorlage.csv", index=False, encoding="utf-8-sig")
vorlage[["lfd_nr", "cluster_id"]].assign(name=vorlage["cluster_id"].map(NAME_JE_CLUSTER)).to_csv(
    AUSGABE / "sichtpruefung_namen_verdeckt.csv", index=False, encoding="utf-8-sig")
print("Geschrieben:", AUSGABE / "sichtpruefung_vorlage.csv", "und", AUSGABE / "sichtpruefung_namen_verdeckt.csv")

#Die Urteile, die kapitel7.ipynb liest, gelten nur für Cluster mit denselben Mitgliedern
if P_SICHTPRUEFUNG.exists():
    urteile = pd.read_csv(P_SICHTPRUEFUNG)
    passend = sum(1 for _, r in urteile.iterrows()
                  if {f.strip() for f in str(r["mitglieder"]).split(",") if f.strip()}
                  == set(MITGLIEDER.get(int(r["cluster_id"]), [])))
    print("Urteile in", P_SICHTPRUEFUNG, "passend zur Partition:", passend, "von", len(urteile), "Clustern")
    print("Verteilung Frage 1:", dict(sorted(Counter(urteile["frage1"].astype(str).str.strip().str.upper()).items())))

Geschrieben: ergebnisse_kapitel7_ap1/sichtpruefung_vorlage.csv und ergebnisse_kapitel7_ap1/sichtpruefung_namen_verdeckt.csv


## Übergabe an kapitel7.ipynb

In [14]:
#Werte, die kapitel7.ipynb fest einträgt, mit ihrer Herkunft in diesem Notebook
pd.DataFrame([
    {"kapitel7.ipynb": "MIN_BELEGE_KERN", "Wert": MIN_BELEGE, "Herkunft": "hergeleitet: Raster der Mindesthäufigkeit"},
    {"kapitel7.ipynb": "LIWC_MIN_STAMMLAENGE", "Wert": LIWC_MIN_STAMMLAENGE, "Herkunft": "gesetzt"},
    {"kapitel7.ipynb": "Ressource", "Wert": RESSOURCE, "Herkunft": "hergeleitet: Abdeckungsvergleich"},
    {"kapitel7.ipynb": "MAX_SYNSETGROESSE", "Wert": MAX_SYNSETGROESSE, "Herkunft": "gesetzt"},
    {"kapitel7.ipynb": "Repräsentation", "Wert": GEWAEHLT, "Herkunft": "hergeleitet: AUC-Vergleich mit Zufallsanker"},
    {"kapitel7.ipynb": "Verfahren", "Wert": LINKAGE, "Herkunft": "hergeleitet: Verfahrenswahl"},
    {"kapitel7.ipynb": "Clusterzahl", "Wert": K, "Herkunft": "hergeleitet: höchste zulässige Silhouette"},
    {"kapitel7.ipynb": "SCHNITTPUNKTE", "Wert": SCHNITTPUNKTE, "Herkunft": "gesetzt"},
    {"kapitel7.ipynb": "MAX_SINGLETON_ANTEIL", "Wert": MAX_SINGLETON_ANTEIL, "Herkunft": "gesetzt"},
    {"kapitel7.ipynb": "MAX_GROESSTES_CLUSTER", "Wert": MAX_GROESSTES_CLUSTER, "Herkunft": "gesetzt"},
    {"kapitel7.ipynb": "MIN_CLUSTER, MAX_CLUSTER", "Wert": (MIN_CLUSTER, MAX_CLUSTER), "Herkunft": "gesetzt"},
    {"kapitel7.ipynb": "MIN_MITGLIEDER_FUER_RESSOURCE", "Wert": MIN_MITGLIEDER_FUER_RESSOURCE, "Herkunft": "gesetzt"},
    {"kapitel7.ipynb": "MIN_DECKUNG_RESSOURCE", "Wert": MIN_DECKUNG_RESSOURCE, "Herkunft": "gesetzt"},
    {"kapitel7.ipynb": "MEHRHEIT_N", "Wert": MEHRHEIT_N, "Herkunft": "gesetzt"},
]).astype({"Wert": str})

,kapitel7.ipynb,Wert,Herkunft
0,MIN_BELEGE_KERN,2,hergeleitet: Raster der Mindesthäufigkeit
1,LIWC_MIN_STAMMLAENGE,3,gesetzt
2,Ressource,OdeNet 1.4,hergeleitet: Abdeckungsvergleich
3,MAX_SYNSETGROESSE,8,gesetzt
4,Repräsentation,B Wortvektoren,hergeleitet: AUC-Vergleich mit Zufallsanker
5,Verfahren,ward,hergeleitet: Verfahrenswahl
6,Clusterzahl,59,hergeleitet: höchste zulässige Silhouette
7,SCHNITTPUNKTE,120,gesetzt
8,MAX_SINGLETON_ANTEIL,0.25,gesetzt
9,MAX_GROESSTES_CLUSTER,0.25,gesetzt
